This notebook records the cells used for the completed development run. Results are archived in `../reports/development_20260906/`; see `REVIEW.md` there and `../NEXT_CHAT_HANDOFF.md` for current findings. The next task is the missing Phase 4 semantic/localization smoke, not rerunning completed gates.


# Phase 4 entry: existing-map smoke and Phase 3 evidence bundle

Run both code cells, in order, in the same Kaggle kernel. All commands execute from `/kaggle/working/newpipeline/projects/logit_evidence_routing`. Cell 1 fetches and verifies the latest `feat/iclr` branch. No model or dataset is downloaded, no VLM is loaded, and no probe is trained. This is a bounded cache-reuse smoke, not the complete Phase 4 experiment.

The default paths match the completed runs supplied in this conversation. If you restored the same artifacts elsewhere, edit the four artifact paths at the top of Cell 1; preserve their contents and identities. Internet is needed for Git fetch; a GPU is not needed for these two cells.

Cell 1 verifies the supplied Phase 3 run and writes `/kaggle/working/phase3_review_bundle.zip`, containing only small metrics, configurations and figures. Cell 2 selects the smallest development-training image ID, joins corrected Phase 1 score maps to Phase 2 spatial metadata, checks identity, geometry and rankings, and measures equal-K localization and agreement. It uses the existing repository metric functions. The smoke produces 14 localization rows and 42 unordered pairwise agreement rows across K=16 and K=32; random seeds are 0/1/2.

Download and attach both ZIP bundles for the next review. The Phase 3 bundle enables full-precision attribute/group analysis. The Phase 4 bundle establishes whether the old maps can be reused without re-extracting the VLM. Neither bundle contains model weights, representation tensors or original images.


In [ ]:
# Cell 1: update source and package the small Phase 3 result files.
import os, sys, json, hashlib, subprocess, zipfile
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
from IPython.display import display, FileLink

PROJECT = Path('/kaggle/working/newpipeline/projects/logit_evidence_routing')
os.chdir(PROJECT)
PHASE3 = Path('/kaggle/working/phase3_development_1a6e0c96681f_20260906T171249251219Z')
CACHE2 = Path('/kaggle/working/phase2_stage_cache')
CACHE1 = Path('/kaggle/working/phase1b_corrected/cache')
GATE1 = Path('/kaggle/working/phase1b_corrected/results/phase1_gate.json')
DIGEST = '63cf0e80ec0a24533682467b6f3b23ccded8625ef25d2fb43d012aa0e72179d8'
STAGES = ['vision.early','vision.middle','vision.late','vision.final','projector.output',
          'llm.early','llm.middle','llm.late','llm.final']
CONTROLS = ['primary','prevalence','shuffled_labels','random_projection']

def require(condition, message):
    if not condition:
        raise RuntimeError(message)
def read_json(path):
    return json.loads(Path(path).read_text())
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def git(*args):
    return subprocess.check_output(['git', *args], text=True).strip()

require(not git('status','--porcelain','--untracked-files=no'), 'Preserve tracked edits before updating.')
subprocess.run(['git','fetch','--no-tags','origin',
                '+refs/heads/feat/iclr:refs/remotes/origin/feat/iclr'], check=True)
subprocess.run(['git','switch','feat/iclr'], check=True)
subprocess.run(['git','merge','--ff-only','origin/feat/iclr'], check=True)
HEAD = git('rev-parse','HEAD')
require(HEAD == git('rev-parse','origin/feat/iclr'), 'Checkout differs from latest fetched branch.')
os.environ['PYTHONPATH'] = 'src'
sys.path.insert(0, str(PROJECT / 'src'))
print('Latest fetched feat/iclr:', HEAD)

report = read_json(PHASE3 / 'phase3_run_report.json')
aggregate = read_json(PHASE3 / 'phase3_aggregation_report.json')
cfg3 = read_json(PHASE3 / 'evaluation_config.json')
for doc in [report, aggregate]:
    require(doc['status']=='PASS' and doc['result_rows']==90 and doc['per_attribute_rows']==2340
            and doc['official_test_images_used']==0 and doc['cache_config_digest']==DIGEST,
            'Phase 3 completion gate differs.')
require(report['stages']==STAGES and report['pooling']==['mean'] and report['seeds']==[0,1,2]
        and report['controls']==CONTROLS and report['selected_attributes']==26
        and report['images']==240 and report['train_images']==160 and report['validation_images']==80,
        'Phase 3 run scope differs.')
require(cfg3['git_commit']==aggregate['git_commit']=='1a6e0c96681f250659ce703207931289cd9112c2'
        and cfg3['epochs']==300 and cfg3['device']=='cuda' and cfg3['random_projection_dim']==256,
        'This is not the supplied completed Phase 3 run.')
summary = pd.read_csv(PHASE3 / 'attribute_probe_by_stage.csv')
attrs = pd.read_csv(PHASE3 / 'per_attribute_metrics.csv')
keys = ['stage','pooling','control','seed']
expected_keys = {(s,'mean',c,seed) for s in STAGES for c in CONTROLS
                 for seed in ([-1] if c=='prevalence' else [0,1,2])}
require(len(summary)==90 and not summary.duplicated(keys).any() and
        set(summary[keys].itertuples(index=False,name=None))==expected_keys, 'Wrong summary keys.')
require(len(attrs)==2340 and not attrs.duplicated(keys+['attribute_id']).any() and
        set(attrs[keys].itertuples(index=False,name=None))==expected_keys and
        attrs.groupby(keys).size().eq(26).all(), 'Wrong attribute keys.')
for table, cols in [(summary,['macro_auroc','macro_f1']), (attrs,['auroc','f1'])]:
    require(np.isfinite(table[cols].to_numpy()).all(), 'Nonfinite metric.')
require(np.isfinite(attrs.loc[attrs.control.ne('prevalence'),'train_loss']).all(), 'Nonfinite fitted loss.')
required = ['phase3_run_report.json','phase3_aggregation_report.json','evaluation_config.json',
            'attribute_probe_by_stage.csv','per_attribute_metrics.csv','stage_seed_summary.csv',
            'paired_seed_deltas.csv','paired_delta_summary.csv']
optional = ['invocation.json','stage_trajectory.png','stage_trajectory.pdf',
            'auroc_trajectory_readable.csv','f1_trajectory_readable.csv']
bundle = Path('/kaggle/working/phase3_review_bundle.zip')
manifest = {}
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for name in required + [name for name in optional if (PHASE3/name).is_file()]:
        path = PHASE3/name
        require(path.is_file() and path.stat().st_size < 25_000_000, f'Missing or oversized report: {name}')
        z.write(path,arcname=name)
        manifest[name] = {'bytes':path.stat().st_size,'sha256':sha256(path)}
    z.writestr('bundle_manifest.json',json.dumps(manifest,indent=2))
print('Phase 3 evidence bundle:', bundle, 'bytes:', bundle.stat().st_size)
display(FileLink(str(bundle)))


In [ ]:
# Cell 2: one development-training image; cached maps only, no VLM/probe run.
os.chdir('/kaggle/working/newpipeline/projects/logit_evidence_routing')
import itertools
import torch
from lger.localization import patch_centers_in_box, selection_localization_metrics, selection_part_metrics
from lger.scoring import stable_topk
from lger.phase1b import feature_key
from lger.stage_cache import config_digest

gate = read_json(GATE1)
cfg1 = read_json(CACHE1 / 'extraction_config.json')
cfg2 = read_json(CACHE2 / 'run_config.json')
index = read_json(CACHE2 / 'index.json')
validation = read_json(CACHE2 / 'validation_report.json')
require(gate['passed'] is True and gate['status'] in ['PASS','PASS WITH ANOMALY']
        and not gate['blocking_findings'], 'Phase 1 gate is not passed.')
require(cfg1['concept_tokenization_policy']=='single_lexical_token_v1'
        and cfg1['concept_token_ids']==[11199,17952], 'Corrected bird/birds cache required.')
require(config_digest(cfg2)==DIGEST==index['config_digest']==validation['config_digest']
        and index['complete'] is True and validation['status']=='PASS'
        and cfg2['official_test_images']==validation['official_test_images']==0
        and validation['official_test_split_untouched'] is True, 'Phase 2 cache gate differs.')
for key in ['model','resolved_revision','quantization','prompt']:
    require(cfg1[key]==cfg2[key], f'Phase 1/2 provenance mismatch: {key}')
require(cfg3['cache_validation_sha256']==sha256(CACHE2/'validation_report.json'),
        'Phase 2 validation report changed since Phase 3.')
require(len(index['records'])==240 and len({r['image_id'] for r in index['records']})==240,
        'Phase 2 index is not the frozen 240-image set.')
require(sum(r['development_split']=='train' for r in index['records'])==160
        and sum(r['development_split']=='val' for r in index['records'])==80, 'Split changed.')
# Fixed choice independent of localization or attribute performance.
chosen = min((r for r in index['records'] if r['development_split']=='train'), key=lambda r:r['image_id'])
image_id = int(chosen['image_id'])
shard = index['shards'][chosen['shard_index']]
metadata_path = CACHE2 / shard['metadata_path']
require(sha256(metadata_path)==shard['metadata_sha256'], 'Shard metadata hash mismatch.')
metadata = read_json(metadata_path)
require(metadata['complete'] is True and metadata['config_digest']==DIGEST, 'Shard metadata differs.')
matches = [item['packed_record'] for item in metadata['records']
           if int(item['packed_record']['image']['image_id'])==image_id]
require(len(matches)==1, 'Missing or repeated smoke record.')
record2 = matches[0]
image2 = record2['image']
require(image2['official_split']=='train' and image2['development_split']=='train',
        'Smoke record is not development training.')
require(len(set(image2['selected_attribute_ids']))==26, 'Attribute subset differs.')
require(set(attrs.attribute_id)==set(image2['selected_attribute_ids']), 'Phase 3 attributes differ from cache.')
path1 = CACHE1 / 'records' / f'{image_id:05d}.pt'
require(path1.is_file(), f'Corrected score-map record unavailable: {path1}; do not substitute the old cache.')
# Load only our own trusted Phase 1 cache. This does not fit a probe or load a VLM.
record1 = torch.load(path1,map_location='cpu',weights_only=False)
require(record1['schema_version']==2 and int(record1['image_id'])==image_id, 'Phase 1 record differs.')
for a,b in [('relative_path','relative_path'),('split','development_split'),
            ('label','class_id'),('class_name','class_name')]:
    require(record1[a]==image2[b], f'Image metadata mismatch: {a}')
spatial = record2['spatial']
grid = tuple(spatial['grid_size'])
size = tuple(spatial['processed_image_size_hw'])
require(tuple(record1['grid_size'])==grid and tuple(record1['processed_image_size'])==size
        and tuple(record1['original_image_size'])==tuple(spatial['original_image_size_wh'])
        and record1['patch_count']==spatial['patch_count']==576, 'Patch geometry differs.')
require(np.allclose(record1['bbox_xyxy_model'],image2['bbox_model_xyxy'],atol=1e-3,rtol=0),
        'Mapped boxes differ; do not join these caches without a geometry audit.')
layer_indices = {name:meta['hidden_state_index'] for name,meta in record2['stage_metadata'].items()}
require(set(layer_indices)==set(STAGES), 'Stage metadata differs.')
print('Cached hidden-state indices:',layer_indices)
print('Projector input stage: vision.late; vision.final is a separate diagnostic checkpoint.')
print('Smoke image:',image_id,'split: development train; official test used: 0')
points = [tuple(p['model_xy']) for p in image2['parts'] if p['visible'] and p['model_xy'] is not None]
bbox_mask = patch_centers_in_box(grid,size,tuple(image2['bbox_model_xyxy']))
methods = ['vision_cls_attention','llm_attention','logit_concept','attention_logit_fusion']
scores = {}
for method in methods:
    score = record1['score_maps'][method].detach().cpu().float()
    require(score.shape==(576,) and torch.isfinite(score).all().item(), f'Invalid scores: {method}')
    require(torch.unique(score).numel()>1, f'Constant score map: {method}')
    scores[method] = score
rows, agreements = [], []
for k in [16,32]:
    selected = {method:stable_topk(score,k) for method,score in scores.items()}
    for method,indices in selected.items():
        require(torch.equal(indices,record1['selections'][feature_key(method,k)].cpu()),
                f'Cached ranking differs: {method}, K={k}')
    for seed in [0,1,2]:
        generator = torch.Generator().manual_seed(seed*1_000_003+image_id)
        selected[f'random_seed_{seed}'] = torch.randperm(576,generator=generator)[:k]
    for method,indices in selected.items():
        metrics = selection_localization_metrics(indices,bbox_mask)
        part_metrics = selection_part_metrics(indices,points,grid_size=grid,image_size=size) if points else {}
        require(all(np.isfinite(value) for value in [*metrics.values(),*part_metrics.values()]),
                f'Nonfinite defined localization metric: {method}')
        rows.append(dict(image_id=image_id,split='train',method=method,K=k,
                         visible_in_crop_part_instances=len(points),**metrics,**part_metrics))
    for a,b in itertools.combinations(selected,2):
        sa,sb = set(selected[a].tolist()),set(selected[b].tolist())
        agreements.append(dict(image_id=image_id,K=k,selector_a=a,selector_b=b,
                               jaccard=len(sa & sb)/len(sa | sb)))
require(len(rows)==14 and len(agreements)==42, 'Unexpected smoke row counts.')
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT4 = Path(f'/kaggle/working/phase4_cached_localizer_smoke_{HEAD[:12]}_{stamp}')
OUT4.mkdir(exist_ok=False)
metrics = pd.DataFrame(rows)
agreement = pd.DataFrame(agreements)
metrics.to_csv(OUT4/'one_image_localization.csv',index=False)
agreement.to_csv(OUT4/'one_image_selector_agreement.csv',index=False)
smoke_report = {'status':'PASS','scope':'one_training_image_existing_cached_localizers_only',
    'phase4_complete':False,'git_commit':HEAD,'cache_config_digest':DIGEST,'image_id':image_id,
    'official_test_images_used':0,'K':[16,32],'random_seeds':[0,1,2],
    'metric_rows':len(rows),'agreement_rows':len(agreements),'stage_indices':layer_indices,
    'part_metrics_available':bool(points),'source_record_sha256':sha256(path1),
    'missing_for_full_phase4':['dense image-text similarity','attribute-specific descriptions and part mapping',
                             'complete matched development evaluation and qualitative review']}
(OUT4/'phase4_cached_localizer_smoke_report.json').write_text(json.dumps(smoke_report,indent=2))
display(metrics)
display(agreement)
print('Cached-localizer smoke PASS:',OUT4)
print('This is a reuse/geometry check, not a Phase 4 result or a VLM utilization test.')
print('No eligible visible parts on this image; part metrics are unavailable.' if not points else
      'Part recall counts distinct part-containing patches; top-1 distance is in patch units.')
bundle4 = Path('/kaggle/working/phase4_cached_localizer_smoke_bundle.zip')
with zipfile.ZipFile(bundle4,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT4.iterdir()):
        z.write(path,arcname=path.name)
display(FileLink(str(bundle4)))


# Interpreting this gate

`phase4_cached_localizer_smoke_report.json` may report PASS only for the one-image reuse/geometry check; `phase4_complete` remains false. A one-image smoke score has no scientific comparison value. The smoke includes Vision-CLS attention, LLM attention, corrected bird/birds Logit Lens, attention/concept fusion, and three random selections. The generic bird/birds map is an object-semantic control, not a map for crown colour or eye stripes.

If the corrected `.pt` record is missing, restore that existing Kaggle artifact; do not silently substitute the uncorrected cache. If geometry differs, inspect preprocessing before joining maps and parts. If there are no visible in-crop parts for the fixed smoke image, part metrics are explicitly unavailable rather than scored as zero.

The existing `part_patch_recall` counts distinct patches containing visible landmarks. `top1_nearest_part_distance_patches` is distance from the top-scoring patch center to the nearest eligible landmark in patch-grid units. Box metrics and part metrics serve different purposes; neither measures whether the VLM uses evidence to answer.

Before the full Phase 4 validation run, add and smoke-test the missing dense image–text similarity method with a verified paired embedding space. Freeze descriptions for the 26 existing attributes and the relevant-part mapping and eligibility rules. Do not compare raw vision and arbitrary text embeddings, treat generic bird scores as attribute-specific scores, or sum independent token marginals to score multi-token phrases. Keep equal K, the same eligible images, and explicit denominators across selectors.

Read `PHASE3_REVIEW_AND_PHASE4_HANDOFF.md` for the full evidence review, topology correction, and scope. The projector consumes the stage labeled `vision.late`; `vision.final` is a separate diagnostic checkpoint. Retain the official CUB test split for the final frozen protocol, and complete localization and utilization phases before selecting a causal intervention.

Local verification: both cells were syntax-checked. The cache-join and localization logic was exercised with tiny synthetic records and score maps, including rejection of an official-test record, incompatible box geometry, and a constant map. Kaggle execution, real cached-map compatibility, and the notebook's pandas/display/export path have not been executed here. No actual extraction or probe experiment ran locally.
